Задание 1 (синтаксис)

Напишите функцию check_car_id(car_id: str) → str.

 Формат номера: 1 буква + 3 цифры + 2 буквы + 2–3 цифры региона.
 Буквы только из набора, допустимого в РФ-номерах: А, В, Е, К, М, Н, О, Р, С, Т, У, Х (кириллица). Учитывайте путаницу с латиницей на входе - нормализуйте или явно опишите правило.

 • Если номер валиден: «Номер А222ВС валиден. Регион: 96.»
 • Если невалиден: «Номер не валиден.»

Пример вывода:

>>> check_car_id('А222ВС96')
 Номер А222ВС валиден. Регион: 96.

 >>> check_car_id('А123ВЕ777')
 Номер А123ВЕ валиден. Регион: 777.

 >>> check_car_id('АБ22ВВ193')
 Номер не валиден.

Пограничные случаи:

>>> check_car_id('A222BC96')   # латиница A/B/C - по правилу нормализации
 Номер А222ВС валиден. Регион: 96.
 # либо «Номер не валиден.» - если нормализацию не делаете; зафиксируйте выбор

 >>> check_car_id('А222ВС9')    # регион из 1 цифры
 Номер не валиден.

 >>> check_car_id('А222ВС9699') # регион из 4 цифр
 Номер не валиден.

 >>> check_car_id('Я222ВС96')   # недопустимая буква Я
 Номер не валиден.

 >>> check_car_id('')
 Номер не валиден.

Задание 2 (повышенная сложность)

Дан многострочный лог. Реализуйте parse_logs(text: str) → list[dict].

 1. Одной (или составной) регуляркой извлеките строки только в формате:
    YYYY-MM-DD HH:MM:SS LEVEL ...
 2. Из хвоста строки достаньте user, action, ip; если есть card=... - тоже.
 3. Валидация через regex:
    • email (разумный практический паттерн);
    • IPv4;
    • карта: маска XXXX-XXXX-XXXX-XXXX; в результате хранить только ****-****-****-1111 (последние 4 цифры).
 4. Невалидные поля не роняют парсер: в словаре "errors": ["invalid_email", ...].
 5. Бонус: удалите последовательные повторы слов в текстовом поле сообщения через re.

Входной пример:

2024-03-15 14:22:01 INFO user=ivan@mail.ru action=login ip=192.168.0.12
 2024-03-15 14:22:05 WARN user=ivan@mail.ru action=retry ip=192.168.0.12
 2024-03-15 14:23:11 ERROR user=bad..email@ action=pay ip=999.1.1.1 card=4111-1111-1111-1111
 Mar 15 2024 14:24:00 INFO user=olga@company.com action=logout ip=10.0.0.5

Пример вывода:

>>> parse_logs(text)
 [
   {
     'ts': '2024-03-15 14:22:01',
     'level': 'INFO',
     'user': 'ivan@mail.ru',
     'action': 'login',
     'ip': '192.168.0.12',
     'card': None,
     'errors': []
   },
   {
     'ts': '2024-03-15 14:22:05',
     'level': 'WARN',
     'user': 'ivan@mail.ru',
     'action': 'retry',
     'ip': '192.168.0.12',
     'card': None,
     'errors': []
   },
   {
     'ts': '2024-03-15 14:23:11',
     'level': 'ERROR',
     'user': 'bad..email@',
     'action': 'pay',
     'ip': '999.1.1.1',
     'card': '****-****-****-1111',
     'errors': ['invalid_email', 'invalid_ip']
   }
 ]
 # Строка «Mar 15 2024 ...» пропускается - формат даты не YYYY-MM-DD.

Пограничные случаи:

• Пустая строка / только пробелы → [].
 • Строка без user=/action=/ip= → запись с отсутствующими полями (None) и соответствующими errors, либо пропуск - зафиксируйте правило.
 • ip=256.0.0.1 или ip=1.2.3 → errors содержит invalid_ip.
 • card=4111111111111111 без дефисов → не распознаётся как карта (card=None) или invalid_card — зафиксируйте правило.
 • Повторы слов в сообщении: «error error error timeout» → «error timeout» (бонус).

Чек-лист самопроверки

Задание 1. Функция check_car_id для валидного номера возвращает сообщение с номером и регионом, иначе - «Номер не валиден.»

Задание 2. Функция parse_logs возвращает список словарей с полями лога и errors; строки неподходящего формата даты пропускаются.

В ЛМС прикреплена ссылка на выполненное задание в Google Colab или GitHub (если вы использовали Jupyter Notebook)

In [5]:
import re

ALPHABET_RU = "АВЕКМНОРСТУХ"  # кириллица
ALPHABET_EN = "ABEKMHOPCTYX"  # латинские двойники

# Правило нормализации: латинские буквы, похожие на допустимые
# кириллические, заменяются на кириллицу; пробелы удаляются.
EN_TO_RU = str.maketrans(ALPHABET_EN, ALPHABET_RU)

PLATE_RE = re.compile(
    rf"(?P<number>[{ALPHABET_RU}]\d{{3}}[{ALPHABET_RU}]{{2}})"
    rf"(?P<region>\d{{2,3}})"
)


def normalize_ru_plate(plate: str) -> str:
    return re.sub(r"\s+", "", plate).upper().translate(EN_TO_RU)


def check_car_id(car_id: str) -> str:
    match = PLATE_RE.fullmatch(normalize_ru_plate(car_id))
    if match:
        return f"Номер {match['number']} валиден. Регион: {match['region']}."
    return "Номер не валиден."


for plate in ["А222ВС96", "А123ВЕ777", "АБ22ВВ193", "A222BC96",
              "А222ВС9", "А222ВС9699", "Я222ВС96", ""]:
    print(f"check_car_id({plate!r})", check_car_id(plate))

check_car_id('А222ВС96') Номер А222ВС валиден. Регион: 96.
check_car_id('А123ВЕ777') Номер А123ВЕ валиден. Регион: 777.
check_car_id('АБ22ВВ193') Номер не валиден.
check_car_id('A222BC96') Номер А222ВС валиден. Регион: 96.
check_car_id('А222ВС9') Номер не валиден.
check_car_id('А222ВС9699') Номер не валиден.
check_car_id('Я222ВС96') Номер не валиден.
check_car_id('') Номер не валиден.


In [6]:
import re
from datetime import datetime

LINE_RE = re.compile(
    r"""
    ^[ \t]*
    (?P<ts>
        \d{4}-(?:0[1-9]|1[0-2])-(?:0[1-9]|[12]\d|3[01])   # дата
        [ ]
        (?:[01]\d|2[0-3]):[0-5]\d:[0-5]\d                 # время
    )
    [ \t]+
    (?P<level>(?i:DEBUG|INFO|WARNING|WARN|ERROR|CRITICAL))\b
    (?:[ \t]+(?P<rest>.*?))?
    [ \t]*$
    """,
    re.MULTILINE | re.VERBOSE,
)

FIELD_RE = re.compile(
    r"(?<![\w.])(?P<key>user|action|ip|card)[ \t]*=[ \t]*(?P<value>\S+)",
    re.IGNORECASE,
)

MSG_RE = re.compile(r"(?<![\w.])msg[ \t]*=[ \t]*(?P<msg>.*?)(?=[ \t]+\w+[ \t]*=|$)")

EMAIL_RE = re.compile(
    r"""
    [A-Za-z0-9_%+-]+(?:\.[A-Za-z0-9_%+-]+)*
    @
    (?:[A-Za-z0-9](?:[A-Za-z0-9-]{0,61}[A-Za-z0-9])?\.)+
    [A-Za-z]{2,}
    """,
    re.VERBOSE,
)

OCTET = r"(?:25[0-5]|2[0-4]\d|1\d\d|[1-9]?\d)"
IPV4_RE = re.compile(rf"{OCTET}(?:\.{OCTET}){{3}}")

CARD_RE = re.compile(r"\d{4}-\d{4}-\d{4}-(?P<last4>\d{4})")

REPEAT_RE = re.compile(r"\b(\w+)(?:\s+\1\b)+", re.IGNORECASE)


def dedupe_words(text: str) -> str:
    return REPEAT_RE.sub(r"\1", text)


def _is_real_date(ts: str) -> bool:
    try:
        datetime.strptime(ts, "%Y-%m-%d %H:%M:%S")
        return True
    except ValueError:
        return False


def parse_logs(text: str) -> list[dict]:
    result = []

    for line in LINE_RE.finditer(text):
        ts = line["ts"]
        if not _is_real_date(ts):
            continue

        rest = line["rest"] or ""

        fields = {}
        for m in FIELD_RE.finditer(rest):
            fields.setdefault(m["key"].lower(), m["value"])

        record = {
            "ts": ts,
            "level": line["level"].upper(),
            "user": fields.get("user"),
            "action": fields.get("action"),
            "ip": fields.get("ip"),
            "card": None,
            "errors": [],
        }
        errors = record["errors"]

        if record["user"] is None:
            errors.append("missing_user")
        elif not EMAIL_RE.fullmatch(record["user"]):
            errors.append("invalid_email")

        if record["action"] is None:
            errors.append("missing_action")

        if record["ip"] is None:
            errors.append("missing_ip")
        elif not IPV4_RE.fullmatch(record["ip"]):
            errors.append("invalid_ip")

        if "card" in fields:
            card = CARD_RE.fullmatch(fields["card"])
            if card:
                record["card"] = f"****-****-****-{card['last4']}"
            else:
                errors.append("invalid_card")

        msg = MSG_RE.search(rest)
        if msg:
            value = msg["msg"].strip().strip('"')
            record["msg"] = dedupe_words(value)

        result.append(record)

    return result


if __name__ == "__main__":
    from pprint import pprint

    sample = """2024-03-15 14:22:01 INFO user=ivan@mail.ru action=login ip=192.168.0.12
 2024-03-15 14:22:05 WARN user=ivan@mail.ru action=retry ip=192.168.0.12
 2024-03-15 14:23:11 ERROR user=bad..email@ action=pay ip=999.1.1.1 card=4111-1111-1111-1111
 Mar 15 2024 14:24:00 INFO user=olga@company.com action=logout ip=10.0.0.5
"""
    pprint(parse_logs(sample), sort_dicts=False)

    print(parse_logs(""), parse_logs("   \n\t\n"))
    print(dedupe_words("error error error timeout"))

    import os
    if os.path.exists("logs_hw4.txt"):
        with open("logs_hw4.txt", encoding="utf-8") as f:
            records = parse_logs(f.read())
        print(f"\nlogs_hw4.txt: {len(records)} записей")


[{'ts': '2024-03-15 14:22:01',
  'level': 'INFO',
  'user': 'ivan@mail.ru',
  'action': 'login',
  'ip': '192.168.0.12',
  'card': None,
  'errors': []},
 {'ts': '2024-03-15 14:22:05',
  'level': 'WARN',
  'user': 'ivan@mail.ru',
  'action': 'retry',
  'ip': '192.168.0.12',
  'card': None,
  'errors': []},
 {'ts': '2024-03-15 14:23:11',
  'level': 'ERROR',
  'user': 'bad..email@',
  'action': 'pay',
  'ip': '999.1.1.1',
  'card': '****-****-****-1111',
  'errors': ['invalid_email', 'invalid_ip']}]
[] []
error timeout

logs_hw4.txt: 583 записей
